# High-performance and parallel computing for AI - JAX sharding tutorial for multi-CPU multi-GPU computing

IMPORTANT
=========

* When using a GPU we need to set some environment variables (see below). If you get some weird error restart the kernel and rerun the code below.
* For these tutorials and practicals we will be using a different `conda environment`. When opening a notebook or a terminal make sure you are using the **hpc4ai Kernel**!!!

Before you start (and before running any other GPU code on the servers) please run the following code, which attempts to limit the maximum GPU memory usage and picks two L40s GPUs at random, excluding the Quadro GPUs. **Please only run the code below once every time you restart the kernel!**

**NOTE:** Here we are setting the `JAX_NUM_CPU_DEVICES` environment variable to tell JAX we want it to see more than one CPU.

**NOTE:** Here we are also defining some helper functions for later. Ignore them.

In [1]:
import os

# JAX-specific environment variables
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.249" # restrict GPU memory preallocation to a fourth of the total
os.environ["JAX_OPTIMIZATION_LEVEL"]="O1"
os.environ["JAX_NUM_CPU_DEVICES"] = "6" # JAX will see 6 CPUs (if available)

## On goliat we have FIVE GPUs so here we pick two of those at random
## so that we do not overload the system.
## The way we do it is by figuring out the GPU UUIDs and then setting
## The CUDA_VISIBLE_DEVICES environment variable.
## Note: this is useful for other libraries as well (e.g., Jax, PyTorch, TF) in multi-GPU servers.

# To get these UUIDs you need to run nvidia-smi -q on the command line
quadro_UUIDs = ["GPU-4efa947b-abbd-7c6e-84f5-61241d34bb4b",
                "GPU-5eb524b0-2b1b-fe98-e6ed-b8fb5185e993"]

L40s_UUIDs = ["GPU-7bba1f33-03d2-016b-d42e-ced83c3ac243",
              "GPU-179d068a-3bea-91d7-1a8c-7017f55d6298",
              "GPU-ae634859-dd49-de46-9182-195639405eaa"]

import random

a, b = random.sample(range(3), 2) # picks two numbers between 0 and 2 at random

# Picks an L40s and a Quadro GPU at random. The others will be invisible to JAX
# NOTE: this only works if the environment variable is set BEFORE JAX is first imported.
os.environ["CUDA_VISIBLE_DEVICES"] = L40s_UUIDs[a] + "," + L40s_UUIDs[b]

## JAX (or whatever other software) will only see these GPUs.

############################# HELPER FUNCTIONS AND IMPORTS #############################

import html
import jax
import jax.numpy as jnp
import ipywidgets as widgets
from IPython.display import display, HTML

gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6 and len(gpus) == 2

def print_sharding_info(x):
    left = widgets.Output(layout=widgets.Layout(
        width='50%',
        border='1px solid #ddd',
        padding='8px'
    ))
    right = widgets.Output(layout=widgets.Layout(
        width='50%',
        border='1px solid #ddd',
        padding='8px'
    ))

    print("\n**Debug info:**\n")
    print("jax.typeof(x): ", jax.typeof(x))
    print("\nx.sharding: ", x.sharding, "\n")

    with left:
        print("jax.debug.visualize_array_sharding(x)\n")
        try: jax.debug.visualize_array_sharding(x)
        except (NotImplementedError,ValueError): print("This visualisation option is not implemented for this type of array!")

    with right:
        lines = []
        lines.append("Using x.addressable_shards:\n")
        lines.append(f"Complete array:\n{x}\n")
        for s in x.addressable_shards:
            lines.append(f"{s.device}\n{s.data}\n")
        text = "\n".join(lines)
        display(HTML(f"<pre style='margin:0; white-space:pre-wrap'>{html.escape(text)}</pre>"))

    display(widgets.HBox([left, right], layout=widgets.Layout(width='100%')))

x = jnp.arange(6 * 4.).reshape(6, 4)
print_sharding_info(x)


**Debug info:**

jax.typeof(x):  float32[6,4]

x.sharding:  SingleDeviceSharding(device=CudaDevice(id=0), memory_kind=device) 



<hr style="height:4px; background-color:black; border:none; border-color:black;">

# Tutorial - JAX array sharding

Here is a simple example in which we shard an array across 6 CPUs using a 3-by-2 mesh.

**Note:** Using more than one CPU in JAX requires setting the environment variable `JAX_NUM_CPU_DEVICES` before JAX is loaded.

In [2]:
# The longer way around
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
pspec = jax.P('X', 'Y')
shrd = jax.NamedSharding(mesh, pspec)
y = jax.device_put(x, shrd)

# Alternatively
jax.set_mesh(mesh)  # explicit mode by default
y = jax.device_put(x, pspec)

# Condensed version
jax.set_mesh(jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus))
y = jax.device_put(x, jax.P('X', 'Y'))

# We will be using a fancier version of this function
def print_sharding_info_debug(x):
    print("\n**Debug info:**\n")
    print("jax.typeof(x): ", jax.typeof(x))
    print("\nx.sharding: ", x.sharding)
    print("\njax.debug.visualize_array_sharding(x)")
    jax.debug.visualize_array_sharding(x)
    print("\nUsing x.addressable_shards:\n")
    print("Complete array:\n", x, "\n")
    for s in x.addressable_shards:
        print(s.device, s.data, sep='\n', end='\n\n')

print_sharding_info(y)


**Debug info:**

jax.typeof(x):  float32[6@X,4@Y]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', 'Y'), memory_kind=device) 



<hr style="height:3px; background-color:black; border:none; border-color:black;">

## Resharding and memory movement

Two key functions allow to move an array across devices and to shard it/change its sharding (also called **resharding**): `jax.device_put` and `jax.reshard`.

In some use cases these are interexchangable. In some others they are not. Here is a summary:

### Tasks only `device_put` can do

- CPU/accelerator (host/device) memory movement.
- Mesh changes.
- Placing values onto a single specific device.
- Transferring JAX pytrees, scalars, and nested Python containers (i.e., more than simple arrays).

### Tasks only `reshard` can do

- Resharding within a function compiled with `jax.jit`.

### Tasks `device_put` and `reshard` are equivalent for

- Sharding/resharding that does not require `jax.jit`, mesh changes, or CPU/accelerator (host/device) memory movement.

### Important

Resharding usually triggers memory movement!

### Resharding by changing sharding

In [3]:
mesh2 = jax.make_mesh((6,), ('X'), devices=cpus)
y = jax.device_put(x, jax.NamedSharding(mesh2, jax.P("X")))
print_sharding_info(y)


**Debug info:**

jax.typeof(x):  float32[6@X,4]

x.sharding:  NamedSharding(mesh=Mesh('X': 6, axis_types=(Explicit,)), spec=P('X',), memory_kind=device) 



### Resharding by using the same mesh, but a different PartitionSpec

In [4]:
y = jax.device_put(x.T, jax.P('Y', 'X'))
print_sharding_info(y)


**Debug info:**

jax.typeof(x):  float32[4@Y,6@X]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('Y', 'X'), memory_kind=device) 



<hr style="height:3px; background-color:black; border:none; border-color:black;">

## Counterintuitive shardings

Some sharding mechanisms are a bit counterintuitive. We now survey them.

Here we go back to the first mesh we defined, namely

```python
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
```

In [5]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

y = jax.device_put(x, jax.P("X", None))
print_sharding_info(y)


**Debug info:**

jax.typeof(x):  float32[6@X,4]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', None), memory_kind=device) 



In [6]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

y = jax.device_put(x, jax.P(None, "Y"))
print_sharding_info(y)


**Debug info:**

jax.typeof(x):  float32[6,4@Y]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P(None, 'Y'), memory_kind=device) 



#### What do you observe?

The array has been copied to the axis omitted with `None`! This is useful in some scenarios, but it may also mean that you have two sets of CPUS running the same exact operations, which is fine, but not great for the environment.

Just so that you are aware, look at what happens if you define an array without using `device_put`:

In [7]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

xx = jnp.ones((5,))
print_sharding_info(xx)


**Debug info:**

jax.typeof(x):  float32[5]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P(None,), memory_kind=device) 



#### What do you see?

The array is copied across all CPUs in the mesh you set with `jax.set_mesh`!

**Note:** This is equivalent to passing `jax.P(None)`.

Again, this is a feature and its fine (except for your electricity bills). If you want to go back to only using a single CPU you can always do:

In [8]:
mesh = jax.make_mesh((1,), ('X'), devices=[cpus[0]])
jax.set_mesh(mesh)

xx = jnp.ones((5,))
print_sharding_info(xx)

mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

yy = jax.device_put(xx, jax.P(None))
print_sharding_info(yy)


**Debug info:**

jax.typeof(x):  float32[5]

x.sharding:  NamedSharding(mesh=Mesh('X': 1, axis_types=(Explicit,)), spec=P(None,), memory_kind=device) 




**Debug info:**

jax.typeof(x):  float32[5]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P(None,), memory_kind=device) 



#### Note that single-device meshes are the default in JAX:
* If JAX sees a GPU it will use a default mesh with only GPU0.
* If it does not see a GPU it will use a default mesh with only CPU0.

### Even more counterintuitive shardings

In [9]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

y = jax.device_put(x, jax.P('X', 'Y'))
print_sharding_info(y)

y = jax.device_put(x, jax.P(('X', 'Y'))) # Note the extra tuple
print_sharding_info(y)


**Debug info:**

jax.typeof(x):  float32[6@X,4@Y]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', 'Y'), memory_kind=device) 




**Debug info:**

jax.typeof(x):  float32[6@(X,Y),4]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P(('X', 'Y'),), memory_kind=device) 



#### What happens?

The array is sharded across both axes $X$ and $Y$ at the same time (i.e., across the whole mesh).

Now, the final option: **Unreducing** mesh axes:

In [10]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

y = jax.device_put(x, jax.P('X', 'Y'))
print_sharding_info(y)

y = jax.device_put(x, jax.P('X', None, unreduced={'Y'}))
print_sharding_info(y)

y = jax.device_put(x, jax.P('X', None))
print_sharding_info(y)


**Debug info:**

jax.typeof(x):  float32[6@X,4@Y]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', 'Y'), memory_kind=device) 




**Debug info:**

jax.typeof(x):  float32[6@X,4]{U:Y}

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', None, unreduced={'Y'}), memory_kind=device) 




**Debug info:**

jax.typeof(x):  float32[6@X,4]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', None), memory_kind=device) 



<hr style="height:3px; background-color:black; border:none; border-color:black;">

## JAX sharding in computations

Once an array is sharded, JAX will handle obvious reductions automatically, e.g.,

In [11]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

y = jax.device_put(x, jax.P('X', 'Y'))
print_sharding_info(y)

z = jnp.sqrt(y) + y
print_sharding_info(z)

s = y.sum()
print_sharding_info(s)

s = y.sum(0)
print_sharding_info(s)

s = y.sum(1)
print_sharding_info(s)


**Debug info:**

jax.typeof(x):  float32[6@X,4@Y]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', 'Y'), memory_kind=device) 




**Debug info:**

jax.typeof(x):  float32[6@X,4@Y]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X', 'Y'), memory_kind=device) 




**Debug info:**

jax.typeof(x):  float32[]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P(), memory_kind=device) 




**Debug info:**

jax.typeof(x):  float32[4@Y]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('Y',), memory_kind=device) 




**Debug info:**

jax.typeof(x):  float32[6@X]

x.sharding:  NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit)), spec=P('X',), memory_kind=device) 



However,

#### JAX will throw an error when reductions are not obvious!

In [12]:
x = jax.device_put(jnp.arange(6 * 4.).reshape(6, 4), jax.P(None, 'Y'))
y = jax.device_put(jnp.arange(4 * 12.).reshape(4, 12), jax.P('Y',None))

print("We want to do a matmul: z = ", jax.typeof(x), " @ ", jax.typeof(y))

try:
  z = jnp.dot(x, y)
except Exception as e:
  print("ERROR!")
  print(e)

We want to do a matmul: z =  float32[6,4@Y]  @  float32[4@Y,12]
ERROR!
Contracting dimensions are sharded and it is ambiguous how the output should be sharded. Please specify the output sharding via the `out_sharding` parameter. Got lhs_contracting_spec=('Y',) and rhs_contracting_spec=('Y',)


#### Why is this not obvious?

It is obvious that `z = x@y` is a $6$-by-$12$ matrix, but it is not obvious how this must be stored across the mesh, so JAX throws an error!

We can specify the output storage layout by providing jax functions with the `out_sharding=` keyword argument:

```python
z = jnp.dot(x, y, out_sharding=sharding)
```

In this case we have multiple options:

In [13]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

x = jax.device_put(jnp.arange(6 * 4.).reshape(6, 4), jax.P(None, 'Y'))
y = jax.device_put(jnp.arange(4 * 12.).reshape(4, 12), jax.P('Y',None))

print("Output is copied across all devices in the mesh")
z = jnp.dot(x, y, out_sharding = jax.P(None))
print("Matmul: ", jax.typeof(z), " = ", jax.typeof(x), " @ ", jax.typeof(y))

print("\n")

print("Output is copied and stored in a similar way as x")
z = jnp.dot(x, y, out_sharding = jax.P(None, "Y"))
print("Matmul: ", jax.typeof(z), " = ", jax.typeof(x), " @ ", jax.typeof(y))

print("\n")

print("Output is copied and stored in a similar way as y")
z = jnp.dot(x, y, out_sharding = jax.P("Y", None))
print("Matmul: ", jax.typeof(z), " = ", jax.typeof(x), " @ ", jax.typeof(y))

print("\n")

print("Output is copied and split along the X mesh axis")
z = jnp.dot(x, y, out_sharding = jax.P("X", None))
print("Matmul: ", jax.typeof(z), " = ", jax.typeof(x), " @ ", jax.typeof(y))
z = jnp.dot(x, y, out_sharding = jax.P(None, "X"))
print("Matmul: ", jax.typeof(z), " = ", jax.typeof(x), " @ ", jax.typeof(y))

print("\n")

print("Output is split across the whole mesh")
z = jnp.dot(x, y, out_sharding = jax.P("X", "Y"))
print("Matmul: ", jax.typeof(z), " = ", jax.typeof(x), " @ ", jax.typeof(y))

#print_sharding_info(x)
#print_sharding_info(y)
#print_sharding_info(z)

Output is copied across all devices in the mesh
Matmul:  float32[6,12]  =  float32[6,4@Y]  @  float32[4@Y,12]


Output is copied and stored in a similar way as x
Matmul:  float32[6,12@Y]  =  float32[6,4@Y]  @  float32[4@Y,12]


Output is copied and stored in a similar way as y
Matmul:  float32[6@Y,12]  =  float32[6,4@Y]  @  float32[4@Y,12]


Output is copied and split along the X mesh axis
Matmul:  float32[6@X,12]  =  float32[6,4@Y]  @  float32[4@Y,12]
Matmul:  float32[6,12@X]  =  float32[6,4@Y]  @  float32[4@Y,12]


Output is split across the whole mesh
Matmul:  float32[6@X,12@Y]  =  float32[6,4@Y]  @  float32[4@Y,12]


#### Important:
* Essentially you can prescribed whatever `out_sharding` you want.
* The output is the same, it is simply copied and/or split in different ways across the devices in the mesh.
* Keep in mind that even if JAX does it automatically for you different PartitionSpecs lead to different (more or less expensive) memory movements!

**NOTE:** Entrywise operations are sharded automatically

You can pass `out_sharding` to many JAX functions, e.g. `jnp.ones`, `jax.random.normal`, `jnp.arange`, etc.

<hr style="height:3px; background-color:black; border:none; border-color:black;">

## Automatic sharding

This all sounds great, but if you have multiple operations it quickly becomes a complete pain since you have to prescribe `out_sharding` for every single operation.

Luckily, JAX has an `@auto_axes` decorator that lets you specify an `out_sharding` only for the function output and automatically sorts out all other shardings.

In [14]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

x = jax.device_put(jnp.arange(6 * 4.).reshape(6, 4), jax.P(None, 'Y'))
y = jax.device_put(jnp.arange(4 * 12.).reshape(4, 12), jax.P('Y',None))
z = jax.device_put(jnp.arange(6 * 12.).reshape(6, 12), jax.P('X', 'Y'))

def fun_by_hand(x,y,z):
    a = jnp.dot(x,y, out_sharding=jax.P("X", "Y")) # Need to specify out_sharding every time
    b = a + z
    c = jnp.dot(a,b.T, out_sharding=jax.P("X", "Y"))
    return c

@jax.sharding.auto_axes
def fun_auto(x,y,z):
    a = jnp.dot(x,y)
    b = a + z
    c = jnp.dot(a,b.T)
    return c

out_by_hand = fun_by_hand(x, y, z)
out_auto1 = fun_auto(x, y, z, out_sharding=jax.P("X", "Y"))
out_auto2 = fun_auto(x, y, z, out_sharding=jax.P())

print(jax.typeof(out_by_hand))
print(jax.typeof(out_auto1))
print(jax.typeof(out_auto2))

float32[6@X,6@Y]
float32[6@X,6@Y]
float32[6,6]


<hr style="height:3px; background-color:black; border:none; border-color:black;">

## Automatic VS Explicit axes

* **Explicit axes.** The default. Require the user to prescribed the `out_sharding` for all ambiguous computations, otherwise an error will be thrown. **Note:** while this sounds like a pain, it makes debugging much easier.
* **Automatic axes.** JAX automatically figures out the `out_sharding` for you (if it can).

What the `@auto_axes` decorator in fact does is temporarily replacing the mesh axes from `Explicit` to `Automatic`.

Rather than using `@auto_axes` it is thus also possible to directly create a mesh with `Automatic` axes as follows:

```python
Auto = jax.sharding.AxisType.Auto
auto_mesh = jax.make_mesh((3, 2), ('X', 'Y'), (Auto, Auto))
jax.set_mesh(auto_mesh)
```

By default this does not happen and `jax.sharding.AxisType.Explicit` axes are used instead.

Here is an example similar to the above one:

In [15]:
Auto = jax.sharding.AxisType.Auto
mesh = jax.make_mesh((3, 2), ('X', 'Y'), (Auto, Auto), devices=cpus)
jax.set_mesh(mesh)

x = jax.device_put(jnp.arange(6 * 4.).reshape(6, 4), jax.P(None, 'Y'))
y = jax.device_put(jnp.arange(4 * 12.).reshape(4, 12), jax.P('Y',None))
z = jax.device_put(jnp.arange(6 * 12.).reshape(6, 12), jax.P('X', 'Y'))

# No need to specify out_sharding every time with Auto axes
a = jnp.dot(x,y)
b = a + z
c = jnp.dot(a,b.T)

# JAX will figure out the output sharding for you, somehow
print(jax.typeof(c))

float32[6,6]


**Note:** The fact that JAX figures it out for you is great, but always check that what JAX decides actually makes sense for what you are trying to achieve

**Important:** You are not allowed to mix explicit and automatic axes, else JAX will throw an error. For instance, the following is wrong

In [16]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

x = jax.device_put(jnp.arange(6 * 4.).reshape(6, 4), jax.P(None, 'Y'))
y = jax.device_put(jnp.arange(4 * 12.).reshape(4, 12), jax.P('Y',None))
z = jax.device_put(jnp.arange(6 * 12.).reshape(6, 12), jax.P('X', 'Y')) # Explicit axes

@jax.sharding.auto_axes
def fun_auto(x,y): # x and y and all arrays defined inside the function will be created with Automatic axes
    a = jnp.dot(x,y)
    b = a + z # z will be taken from outside the function scope. The problem is that z has Explicit axes!!
    c = jnp.dot(a,b.T)
    return c

try: fun_auto(x, y, out_sharding=jax.P("X", "Y"))
except ValueError as e:
    print("ValueError!\n")
    print(e)
    print("\nNote that in the above the two AbstractMeshes being compared have Auto and Explicit axes respectively!")

ValueError!

For primitive add, context mesh AbstractMesh('X': 3, 'Y': 2, axis_types=(Auto, Auto), device_kind=cpu, num_cores=None) should match the aval mesh AbstractMesh('X': 3, 'Y': 2, axis_types=(Explicit, Explicit), device_kind=cpu, num_cores=None) for shape float32[6@X,12@Y]. This error occurs at source:  

Note that in the above the two AbstractMeshes being compared have Auto and Explicit axes respectively!


### Compiler hints with Auto axes

Sometimes it is good to suggest the compiler that it should choose a specific intermediate sharding.

You can do this with `array = jax.lax.with_sharding_constraint(array, sharding)`. For instance

In [17]:
Auto = jax.sharding.AxisType.Auto
mesh = jax.make_mesh((3, 2), ('X', 'Y'), (Auto, Auto), devices=cpus)
jax.set_mesh(mesh)

x = jax.device_put(jnp.arange(6 * 4.).reshape(6, 4), jax.P(None, 'Y'))
y = jax.device_put(jnp.arange(4 * 12.).reshape(4, 12), jax.P('Y',None))
z = jax.device_put(jnp.arange(6 * 12.).reshape(6, 12), jax.P('X', 'Y'))

def make_fun_auto(hint):
    @jax.jit
    def fun_auto(x,y,z):
        a = jnp.dot(x,y)
        b = a + z
        c = jnp.dot(a,b.T)
        c = jax.lax.with_sharding_constraint(c, hint)
        return c

    return fun_auto

out_auto1 = make_fun_auto(jax.P())(x, y, z)
out_auto2 = make_fun_auto(jax.P("X", "Y"))(x, y, z)

print("NOTE THE DIFFERENCE: x.sharding")
print(out_auto1.sharding)
print(out_auto2.sharding)

print("\nNOTE THE DIFFERENCE: jax.typeof")
print(jax.typeof(out_auto1))
print(jax.typeof(out_auto2))

NOTE THE DIFFERENCE: x.sharding
NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Auto, Auto)), spec=P(), memory_kind=device)
NamedSharding(mesh=Mesh('X': 3, 'Y': 2, axis_types=(Auto, Auto)), spec=P('X', 'Y'), memory_kind=device)

NOTE THE DIFFERENCE: jax.typeof
float32[6,6]
float32[6,6]


**NOTE:** When Auto axes are used you cannot trust `jax.typeof` anymore. You need to query `x.sharding` which is the concrete sharding chosen by the compiler!!

**NOTE:** `jax.lax.with_sharding_constraint` can also be used with `Explicit` axes. In this case it is an assertion that throws an error if the sharding is not respected.

In [18]:
mesh = jax.make_mesh((3, 2), ('X', 'Y'), devices=cpus)
jax.set_mesh(mesh)

x = jax.device_put(jnp.arange(6 * 4.).reshape(6, 4), jax.P(None, 'Y'))
y = jax.device_put(jnp.arange(4 * 12.).reshape(4, 12), jax.P('Y',None))
z = jax.device_put(jnp.arange(6 * 12.).reshape(6, 12), jax.P('X', 'Y'))

def fun_by_hand(x,y,z):
    a = jnp.dot(x,y, out_sharding=jax.P("X", "Y")) # Need to specify out_sharding every time
    b = a + z
    c = jnp.dot(a,b.T, out_sharding=jax.P("X", "Y"))
    c = jax.lax.with_sharding_constraint(c, jax.P()) # This will never happen
    return c

try: out_by_hand = fun_by_hand(x, y, z)
except AssertionError as e:
    print("AssertionError!")
    print(e)

AssertionError!
`with_sharding_constraint` acts as an assert when all axes of mesh are of type `Explicit`. The array sharding: P('X', 'Y') did not match the sharding provided: P(None, None). Please use `jax.sharding.reshard` to shard your input to the sharding you want.


<hr style="height:3px; background-color:black; border:none; border-color:black;">

# Advanced topics

## Advanced topic (curiosity): Concrete VS abstract meshes and JIT compilation

* A **concrete mesh** is what we have seen so far. It has a shape, named axes, and an array of devices.
* An **abstract mesh** is a concrete mesh without any info about the specific devices being used.

This would not be important if it weren't for the fact that when `auto_axes` or `Automatic` axes are used with `jax.jit`, we only have access to the abstract mesh during execution.

Again, not a big deal, but this means that we cannot use `jax.get_mesh`. Instead, we must use

```python
abstract_mesh = jax.sharding.get_abstract_mesh()
```

Similarly, you cannot change concrete mesh during a jit-compiled function. You can only change the abstract mesh with
the context manager:

```python
with jax.sharding.use_abstract_mesh(abstract_mesh):
    # DO SOMETHING
```

Abstract meshes are defined similarly as for concrete mesh, but with a different function:

```python
abstract_mesh = jax.sharding.AbstractMesh((3, 2), ('X', 'Y')) # does not take any devices as inputs
```

**To recap:** In JIT-compiled functions using `auto_axes` or `Automatic` axes we lose access to concrete meshes so we need to use abstract meshes instead for which function names are different. The equivalences are:

* `jax.make_mesh` -> `jax.sharding.AbstractMesh`
* `jax.set_mesh` -> `with jax.sharding.use_abstract_mesh(abstract_mesh):`
* `jax.get_mesh` -> `jax.sharding.get_abstract_mesh()`
* Additionally, you can only use `jax.reshard` within compiled blocks (no `jax.device_put`).

## Advanced topic (outside the course scope): Manual sharding

With automatic sharding JAX handles all communication automatically. There is also **Manual sharding**, which forces to/lets the user decide how the devices should communicate to perform operations. Here things like changing the abstract mesh become important since it may change the way communication happens.

**This is outside the scope of this course**, but to read more, see the [official JAX docs](https://docs.jax.dev/en/latest/notebooks/shard_map.html).

<hr style="height:4px; background-color:black; border:none; border-color:black;">